<div align="center">

# 🇳🇵 Nepal GovAgent — Live Demo
### Ask Nepal's government documents anything. In Nepali or English.

[![PyPI](https://img.shields.io/pypi/v/nepal-gov-agent?color=teal)](https://pypi.org/project/nepal-gov-agent/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![GitHub](https://img.shields.io/badge/GitHub-irfanalidv%2FNepal--Gov--Agent-black?logo=github)](https://github.com/irfanalidv/Nepal-Gov-Agent)

**Built by [Irfan Ali](https://github.com/irfanalidv) · DataCortex IQ**

</div>

---

## 📁 What's available — 5 documents via `download_corpus()`

The next step downloads five Nepal government PDFs into `./nepal_gov_data/` (no separate repo checkout). Example questions you can try:

| Document | Language | Example questions |
|---|---|---|
| **National AI Policy 2082** | English | Data privacy · AI governance |
| **Constitution of Nepal** (2nd amendment) | English | Fundamental rights |
| **Digital Nepal Framework 2.0** | English | Digitization priorities |
| **प्रतिनिधि सभा निर्वाचन अध्यादेश २०८२** | Nepali 🇳🇵 | Candidate eligibility · voting |
| **मानव अधिकार पुरस्कार कोष नियमावली** | Nepali 🇳🇵 | Fund purpose · awards |

> 💡 **Add your own PDFs?** See **Use Your Own Nepal Government PDFs** near the end.

---

## How it works

Every answer includes **source citations** — document name and page. Runs offline on this machine; nothing is sent to the cloud.

> ⏱️ **First run:** Install + download takes ~2 minutes. After that, answers are instant.



> ⚠️ **Research Prototype** — This is not production-ready for government deployment. The benchmark measures retrieval quality only, not answer safety or factual correctness. See [Scope and Limitations](https://github.com/irfanalidv/Nepal-Gov-Agent#scope-and-limitations) in the README before any official use.



## 🛠️ Setup — Run once


In [1]:
# nepal-gov-agent pulls retrieval stack + requests for corpus download
!pip install -U "nepal-gov-agent>=0.2.3" --quiet

print("✅ Installation complete")




[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ Installation complete


In [2]:
# ------------------------------------------------------------
# Option A — Download the seed corpus (same PDFs as repo Data/)
# 5 documents: AI Policy, Constitution, Digital Nepal Framework,
# election ordinance (Nepali), human rights fund rules (Nepali).
# (Seed omits the Law Commission Legal Maxims volume — see README.)
# ------------------------------------------------------------
from nepal_gov_agent import GovRAG, GovRAGConfig, download_corpus

corpus_dir = download_corpus()  # downloads to ./nepal_gov_data/
import os

# Older seed downloads included a large Legal Maxims PDF — remove if present
_legacy_maxims = os.path.join(corpus_dir, "1714977234_32.pdf")
if os.path.isfile(_legacy_maxims):
    os.remove(_legacy_maxims)
    print("Removed legacy Legal Maxims PDF from corpus folder")

# ------------------------------------------------------------
# Option B — Use your own Nepal government PDFs
# Comment out Option A above, uncomment below, point at your folder
# ------------------------------------------------------------
# corpus_dir = "./my_ministry_docs/"

_cfg = GovRAGConfig(cache_dir=os.path.join(corpus_dir, ".nepal_gov_cache"))
rag = GovRAG(corpus_dir=corpus_dir, config=_cfg)
print(f"✅ GovRAG ready — corpus: {corpus_dir}")
print("🔒 Fully offline. No data leaves this machine.")



📁 Corpus folder: ./nepal_gov_data
📥 Downloading 5 Nepal government documents...

   ✓ Already exists — skipping: 2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२_v1cs5ms.pdf
   ✓ Already exists — skipping: Constitution of Nepal (2nd amd. English)_xf33zb3.pdf
   ✓ Already exists — skipping: National AI Policy-Final_uxc94vg.pdf
   ✓ Already exists — skipping: dnf_jbji8eb.pdf
   ✓ Already exists — skipping: मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075_n4hme7v.pdf

✅ Seed corpus ready — 5 documents in ./nepal_gov_data
💡 To use your own PDFs, pass any folder to GovRAG(corpus_dir=...)



✅ GovRAG ready — corpus: ./nepal_gov_data
🔒 Fully offline. No data leaves this machine.


## 🔍 Live Demos — Ask Nepal's Government Documents
*Each demo below shows a real business scenario. Run any cell to see live retrieval.*



In [3]:
# Helper — run this once before the demo cells below
from IPython.display import Markdown, display
import re

_DOC_ALIASES = {
    "dnf_jbji8eb.pdf": "Digital Nepal Framework 2.0",
}


def _clean_doc_display(doc_id: str) -> str:
    if doc_id in _DOC_ALIASES:
        return _DOC_ALIASES[doc_id]
    base = doc_id.replace(".pdf", "").replace(".PDF", "")
    if "_" in base:
        prefix, suffix = base.rsplit("_", 1)
        if suffix.isalnum() and len(suffix) >= 5:
            return prefix
    return base


def ask_and_display(question, context="", *, rag_client=None):
    g = rag if rag_client is None else rag_client
    if context:
        display(Markdown(f"> 💼 **Scenario:** {context}"))
    display(Markdown(f"**❓ Question:** `{question}`"))

    result = g.ask(question)

    answer = result.answer or ""
    # Strip retrieval headers like "[file.pdf, p.15] Block 1" (line may be followed by URL or page echo)
    answer = re.sub(r"\[[^\]]*?\]\s*Block\s*\d+\s*\n", "", answer)
    answer = re.sub(
        r"^www\.lawcommission\.gov\.np\s*\n", "", answer, flags=re.MULTILINE
    )
    # Remove standalone page number lines (PDF page echo)
    answer = re.sub(r"^\d+\s*\n", "", answer, flags=re.MULTILINE)
    answer = re.sub(r"\n{3,}", "\n\n", answer).strip()

    if len(answer) > 500:
        truncated = answer[:500]
        last_period = truncated.rfind(".")
        if last_period > 200:
            answer = truncated[: last_period + 1] + "\n\n*[See source documents for full text]*"
        else:
            answer = truncated + "…\n\n*[See source documents for full text]*"

    display(Markdown("---"))
    display(Markdown("**📋 Answer:**"))
    display(Markdown(answer))
    display(Markdown("---"))
    display(Markdown("**📌 Sources:**"))
    for src in result.sources[:3]:
        doc_clean = _clean_doc_display(src["doc"])
        display(Markdown(f"- 📄 `{doc_clean}` — Page {src['page']}"))
    display(Markdown(""))
    return result



---

### Usecase 1 — National AI Policy (English)

**Scenario:** Tech entrepreneur checking data governance before launching in Nepal.

**Question:** *What does Nepal's National AI Policy say about data privacy and sovereignty?*



In [4]:
_ = ask_and_display(
    "What does Nepal's National AI Policy say about data privacy and sovereignty?",
    context="Tech entrepreneur checking data governance before launching in Nepal.",
)



> 💼 **Scenario:** Tech entrepreneur checking data governance before launching in Nepal.

**❓ Question:** `What does Nepal's National AI Policy say about data privacy and sovereignty?`

---

**📋 Answer:**

9.49 Use AI to enhance easier access to public service delivery for women, 
children, senior citizens, marginalized communities, minorities, and 
persons with disabilities. 
9.50 Expand AI use in citizen security, crime investigation and surveillance, and 
emergency response. 
9.51 Maximize AI use for the effective implementation of the Digital Nepal 
Framework. 
9.52 Apply AI to enhance the effectiveness of services delivered through the 
Nagarik App. 
Related to Strategy 8.

*[See source documents for full text]*

---

**📌 Sources:**

- 📄 `National AI Policy-Final` — Page 15

- 📄 `National AI Policy-Final` — Page 20

- 📄 `National AI Policy-Final` — Page 25

---

### Usecase 2 — Constitutional rights (English)

**Scenario:** NGO designing a citizen services program.

**Question:** *What fundamental rights does the Constitution of Nepal guarantee?*



In [5]:
_ = ask_and_display(
    "What fundamental rights does the Constitution of Nepal guarantee?",
    context="NGO designing a citizen services program.",
)



> 💼 **Scenario:** NGO designing a citizen services program.

**❓ Question:** `What fundamental rights does the Constitution of Nepal guarantee?`

---

**📋 Answer:**

(g) 
 In case Nepal has to become a party to any international treaty or 
agreement on human rights, to make recommendation, accompanied 
by the reasons therefor, to the Government of Nepal; and monitor 
whether any such treaty or an agreement to which Nepal is already a 
party has been implemented, and in case it is found not to have been 
implemented, to make recommendation to the Government of Nepal 
for its implementation; 
(h) 
To publish, in accordance with law, the names of the officials,…

*[See source documents for full text]*

---

**📌 Sources:**

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 139

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 165

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 22

---

### Usecase 3 — Election ordinance (Nepali 🇳🇵)

**Scenario:** Municipality officer in Butwal checking candidate eligibility rules.

**Question:** *प्रतिनिधि सभाको निर्वाचनमा उम्मेदवार हुन के-के योग्यता चाहिन्छ?*

*(What qualifications are needed to be a candidate in the House of Representatives election?)*



In [6]:
_ = ask_and_display(
    "प्रतिनिधि सभाको निर्वाचनमा उम्मेदवार हुन के-के योग्यता चाहिन्छ?",
    context="Municipality officer in Butwal checking candidate eligibility rules.",
)



> 💼 **Scenario:** Municipality officer in Butwal checking candidate eligibility rules.

**❓ Question:** `प्रतिनिधि सभाको निर्वाचनमा उम्मेदवार हुन के-के योग्यता चाहिन्छ?`

---

**📋 Answer:**

अिुसूची-१ 
  (दफा २८ को उपदफा (५) र दफा ६० को उपदफा (६) सँग सम्बशरिि) 
उम्मेदर्ारको बरदसूचीको लातग समार्ेशी आिार 
क्र.सं. 
प्रतितितित्र् िुिे समूि 
प्रतिशि 
१. 
दतलि 
13.44 
२. 
आददर्ासी जिजाति 
28.72 
३. 
िस आया 
30.28 
४. 
मिेशी 
16.15 
५. 
थारु 
6.52 
६. 
मुशस्लम 
4.89

---

National Artificial Intelligence (A.I.) Policy, 2025

---

Table of Contents 
Chapter One: Introduction .......................................................................... 4 
1. Background .........................

*[See source documents for full text]*

---

**📌 Sources:**

- 📄 `2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२` — Page 2

- 📄 `National AI Policy-Final` — Page 1

- 📄 `National AI Policy-Final` — Page 2

---

### Usecase 4 — Human Rights Award Fund (Nepali 🇳🇵)

**Scenario:** Human rights researcher understanding fund operations.

**Question:** *मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?*

*(What is the purpose and process of the Human Rights Award Fund?)*

*(This demo scopes retrieval to the **Human Rights Award Fund rules** PDF only so the answer is not conflated with other Nepali gazette texts in the seed set.)*



In [7]:
import os
import shutil

# Other Nepali PDFs in the seed set can out-rank this rules booklet in hybrid search.
# Build a one-document mini-corpus so this cell demonstrates the correct source.
_hr_pdf = next(
    f
    for f in os.listdir(corpus_dir)
    if f.endswith(".pdf") and "मानव अधिकार" in f
)
_u4_dir = os.path.join(corpus_dir, ".demo_u4_corpus")
shutil.rmtree(_u4_dir, ignore_errors=True)
os.makedirs(_u4_dir, exist_ok=True)
shutil.copy2(os.path.join(corpus_dir, _hr_pdf), os.path.join(_u4_dir, _hr_pdf))

_rag_u4 = GovRAG(
    corpus_dir=_u4_dir,
    config=GovRAGConfig(cache_dir=os.path.join(_u4_dir, ".nepal_gov_cache")),
)

_ = ask_and_display(
    "मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?",
    context="Human rights researcher understanding fund operations.",
    rag_client=_rag_u4,
)



> 💼 **Scenario:** Human rights researcher understanding fund operations.

**❓ Question:** `मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?`

---

**📋 Answer:**

मानव अधिकार पुरस्कार कोष सञ्‍चालन धनयमावली, २०७५ 
 
नेपाल‍राजपत्रमा‍प्रकाशित‍धमधत 
2075।09।09 
संिोिन 
1. 
मानव अधिकार पुरस्कार कोष सञ्‍चालन (पहिलो‍संिोिन)‍ 
धनयमावली, २०82 
 
 
 
 
‍‍‍‍‍ ‍‍‍‍2082।09।24 
 
प्रिासकीय  कायहवधि (धनयधमत गने) ऐन, २०१३ को दफा २ ले ददएको अधिकार प्रयोग गरी नेपाल 
सरकारले देिायका धनयमिरु बनाएको छ। 
1. 
संशिप्‍त नाम र प्रारम्भः ‍(१) यी धनयमिरुको नाम "मानव अधिकार पुरस्कार कोष सञ्‍चालन 
धनयमावली, २०७५" रिेको  छ।  
(२) यो धनयमावली तुरुन्त प्रारम्भ िुनेछ।  
२.

*[See source documents for full text]*

---

**📌 Sources:**

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 1

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 2

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 3

---

## ✏️ Try Your Own Question

Type any question below in English or Nepali and run the cell.



In [8]:
# Change this to anything you want to ask
YOUR_QUESTION = "What role does the National AI Centre play in Nepal?"

# Examples in Nepali:
# YOUR_QUESTION = "नेपालको संविधानमा शिक्षाको अधिकार कसरी परिभाषित गरिएको छ?"
# YOUR_QUESTION = "डिजिटल नेपाल फ्रेमवर्कको उद्देश्य के हो?"

_ = ask_and_display(YOUR_QUESTION)



**❓ Question:** `What role does the National AI Centre play in Nepal?`

---

**📋 Answer:**

Related to Strategy 8.3 (Develop advanced, high-capacity infrastructures 
related to AI) 
9.12 Improve existing information and communication technology (ICT) 
infrastructure to make it AI-friendly. 
9.13 Make provision for infrastructures required for AI development and 
application, including Data Centre, Cloud Infrastructure, high-performance 
computing, reliable electricity supply and high-speed internet. 
9.

*[See source documents for full text]*

---

**📌 Sources:**

- 📄 `National AI Policy-Final` — Page 11

- 📄 `National AI Policy-Final` — Page 18

- 📄 `National AI Policy-Final` — Page 20

---

## 📊 Retrieval Validation

These are real benchmark results run against the 5-document seed corpus.
Recall@3 = 0.857 means: for 6 out of 7 test queries, the correct document
section appeared in the top 3 retrieved results.

> This measures **retrieval quality only** — not answer safety or correctness.
> See [Scope and Limitations](https://github.com/irfanalidv/Nepal-Gov-Agent#scope-and-limitations).



In [9]:
import logging

logging.getLogger("nepal_gov_agent").setLevel(logging.WARNING)

from nepal_gov_agent import run_benchmark

results = run_benchmark(rag, verbose=False)
print(results.report())



Nepal GovAgent Benchmark Results
Total queries:      7
Recall@1:           0.714
Recall@3:           0.857
Recall@5:           1.000
Keyword hit rate:   1.000
Doc hit rate:       1.000
Nepali recall@3:    1.000
English recall@3:   0.833


---

## 📂 Use Your Own Nepal Government PDFs

This library works with **any** Nepal government PDF — not just the 5 seed
documents. Ministry circulars, municipal SOPs, land records, provincial
guidelines — drop them in and ask questions.



In [10]:
import os

# Option 1 — Upload from your computer (Google Colab only)
try:
    from google.colab import files

    print("Select your Nepal government PDF(s) to upload...")
    uploaded = files.upload()

    os.makedirs("./my_ministry_docs/", exist_ok=True)
    for fname in uploaded:
        dest = os.path.join("./my_ministry_docs/", fname)
        with open(dest, "wb") as f:
            f.write(uploaded[fname])
        print(f"✅ Added: {fname}")

    print("\n🔄 Now re-run the Setup cell pointing at your folder:")
    print('   rag = GovRAG(corpus_dir="./my_ministry_docs/")')
except ImportError:
    print("Not in Google Colab — copy PDFs into a folder locally, then:")
    print('   rag = GovRAG(corpus_dir="./my_ministry_docs/")')

# Option 2 — Point at a folder you already have
# rag = GovRAG(corpus_dir="./my_ministry_docs/")



Not in Google Colab — copy PDFs into a folder locally, then:
   rag = GovRAG(corpus_dir="./my_ministry_docs/")


---

## 🤝 Contribute

The more documents in the corpus, the more useful this becomes.

**What's needed most:**
- Ministry circulars (2080–2082 BS)
- Provincial government SOPs
- Municipality service guidelines
- Land, labor, tax regulation PDFs

Open a PR: **[github.com/irfanalidv/Nepal-Gov-Agent](https://github.com/irfanalidv/Nepal-Gov-Agent)**

---

<div align="center">

**Built in the spirit of AIAN's four pillars: Data. Infrastructure. Policy. Resources.**

🇳🇵 *Working on Nepal's AI infrastructure layer? Open an issue or reach out.*

**Irfan Ali** · [LinkedIn](https://www.linkedin.com/in/irfanalidv/) ·
[GitHub](https://github.com/irfanalidv) · irfan.ali@datacortex.in

</div>

